In [29]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV , RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df = pd.read_csv('../resources/housing.csv')

In [7]:
X , y = df.iloc[:,:-1] , df.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

rf_pipeline = Pipeline([
    ('st_scaler', StandardScaler()),
    ('rf_model', RandomForestRegressor())
])

scores = cross_val_score(rf_pipeline,X,y,scoring='neg_mean_squared_error',cv=10)

final_avg_rmse = np.mean(np.sqrt(np.abs(scores)))

print('Final RMSE: ',final_avg_rmse)


Final RMSE:  30750.291086266378


# Additional Components introduced for pipelines
- sklearn_pandas : interoperability between pandas and scikit-learn
- sklearn.impute : SimpleImputer Native imputation of numerical and categorical columns in scikit-learn
- sklearn.pipeline : FeatureUnion combine multiple pipelines of features into a single pipeline of feature

# Sklearn Pipeline example with XGBoost

In [19]:
X , y = df.iloc[:,:-1] , df.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

xgb_pipeline = Pipeline([
    ('st_scaler', StandardScaler()),
    ('xgb_model', xgb.XGBRegressor())
])

scores = cross_val_score(xgb_pipeline,X,y,scoring='neg_mean_squared_error',cv=10)

final_avg_rmse = np.mean(np.sqrt(np.abs(scores)))

print('Final RMSE: ',final_avg_rmse)


Final RMSE:  30402.465886275975


# Tuning XGBoost hyperparameters in pipeline


In [32]:
X , y = df.iloc[:,:-1] , df.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

xgb_pipeline = Pipeline([
    ('st_scaler', StandardScaler()),
    ('xgb_model', xgb.XGBRegressor())
])

gbm_param_grid = {
    'xgb_model__subsample' : np.arange(.05,1,.05),
    'xgb_model__max_depth' : np.arange(3,20,1),
    'xgb_model__colsample_bytree' : np.arange(.1,1.05,.05)
}
randomized_neg_mse = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=gbm_param_grid,
    n_iter=10,
    scoring='neg_mean_squared_error',
    cv=4
)

randomized_neg_mse.fit(X,y)

final_avg_rmse = np.mean(np.sqrt(np.abs(randomized_neg_mse.best_score_)))

print('Final RMSE: ',final_avg_rmse)

Final RMSE:  31696.545174513893


# Excercise

In [ ]:
# Import LabelEncoder
from sklearn.preprocessing import LabelEncoder


# Fill missing values with 0
df.LotFrontage = df.LotFrontage.fillna(0)

# Create a boolean mask for categorical columns
categorical_mask = (df.dtypes == 'object')

# Get list of categorical column names
categorical_columns = df.columns[categorical_mask].tolist()

# Print the head of the categorical columns
print(df[categorical_columns].head())

# Create LabelEncoder object: le
le = LabelEncoder()

# Apply LabelEncoder to categorical columns
df[categorical_columns] = df[categorical_columns].apply(lambda x: le.fit_transform(x))

# Print the head of the LabelEncoded categorical columns
print(df[categorical_columns].head())

0
Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]
Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


In [ ]:
# Import necessary modules
from sklearn_pandas import DataFrameMapper
from sklearn.impute import SimpleImputer

# Check number of nulls in each feature column
nulls_per_column = X.isnull().sum()
print(nulls_per_column)

# Create a boolean mask for categorical columns
categorical_feature_mask = X.dtypes == object

# Get list of categorical column names
categorical_columns = X.columns[categorical_feature_mask].tolist()

# Get list of non-categorical column names
non_categorical_columns = X.columns[~categorical_feature_mask].tolist()

# Apply numeric imputer
numeric_imputation_mapper = DataFrameMapper(
                                            [([numeric_feature], SimpleImputer(strategy="median")) for numeric_feature in non_categorical_columns],
                                            input_df=True,
                                            df_out=True
                                           )

# Apply categorical imputer
categorical_imputation_mapper = DataFrameMapper(
                                                [(category_feature, SimpleImputer(strategy="median")) for category_feature in categorical_columns],
                                                input_df=True,
                                                df_out=True
                                               )

# Import FeatureUnion
from sklearn.pipeline import FeatureUnion

# Combine the numeric and categorical transformations
numeric_categorical_union = FeatureUnion([
                                          ("num_mapper", numeric_imputation_mapper),
                                          ("cat_mapper", categorical_imputation_mapper)
                                         ])

ImportError: cannot import name 'tosequence' from 'sklearn.utils' (/Users/apple/Documents Local/Mubashir Code/ML/.venv/lib/python3.12/site-packages/sklearn/utils/__init__.py)